# Lab 9 — LangGraph: State, nodes, and the loop on the edge (interactive)

Open this as a **notebook** (`explore.ipynb`) or run the `.py` cell-by-cell in VSCode with
**Shift+Enter**. You build the graph up from its parts: the typed State and its reducer, one node
at a time, then the conditional edge that makes the write/critique cycle a *declared* loop rather
than a `while`.

Runs with **no key** — `ask()` returns a deterministic `[STUB]` string, so the graph still drives
end to end and every structural lesson lands. With `OPENAI_API_KEY` you get real essays; with
`TAVILY_API_KEY` you get live research instead of the local corpus.

Same code as `essay_writer.py`. The CLI is `python essay_writer.py "topic"`.

**How to work through it** — three beats:
- 🔮 **Predict** — before you run a cell, say what you expect. Then run and compare.
- 🛠️ **Your turn** — let the critic end the loop early.
- 🗣️ **Explain** — end by answering the question in your own words.

In [1]:
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
sys.path.insert(0, HERE)

from essay_writer import (  # noqa: E402
    EssayState, DEFAULT_REVISIONS, MODEL, LLM,
    build_graph, plan_node, research_node, write_node, critique_node,
    should_continue, _local_corpus_search,
)
print("LLM:", f"live {MODEL}" if LLM is not None else "STUB (no OPENAI_API_KEY)")
print("research:", "Tavily (live)" if os.getenv("TAVILY_API_KEY") else "local ./corpus")

LLM: live gpt-4.1
research: local ./corpus


## 1. The State is the contract

Every node takes State and returns a **partial update dict** — nothing else. `notes` is special:
`Annotated[list[str], operator.add]` makes it a **reducer**, so updates *append* instead of
replacing.

In [2]:
EssayState.__annotations__

{'task': str,
 'plan': str,
 'notes': typing.Annotated[list[str], <built-in function add>],
 'draft': str,
 'critique': str,
 'revisions': int,
 'max_revisions': int}

## 2. 🔮 Predict: reducer vs. replace

Two nodes each return `{"notes": ["..."]}`, and a third returns `{"draft": "..."}` twice. After
all of that, how many items are in `notes`, and how many drafts are in `draft`? Say it out loud,
then run.

In [3]:
import operator
from typing import Annotated, get_args, get_origin

for field, ann in EssayState.__annotations__.items():
    reducer = get_args(ann)[1] if get_origin(ann) is Annotated else None
    print(f"{field:<14} {'APPENDS (reducer: ' + reducer.__name__ + ')' if reducer else 'replaced on update'}")

task           replaced on update
plan           replaced on update
notes          APPENDS (reducer: add)
draft          replaced on update
critique       replaced on update
revisions      replaced on update
max_revisions  replaced on update


## 3. One node at a time — a node is just a function

No framework magic: call it with a dict, read the dict it hands back.

In [4]:
state: EssayState = {
    "task": "Why reflection improves agents",
    "plan": "", "notes": [], "draft": "", "critique": "",
    "revisions": 0, "max_revisions": 1,
}
upd = plan_node(state)
print("plan_node returned keys:", list(upd))
print(upd["plan"][:400])

plan: outlining

plan_node returned keys: ['plan']
- Reflection enables agents to analyze past actions, identifying successes and mistakes for continuous improvement.  
- Through reflective processes, agents can adapt strategies to dynamic environments, enhancing flexibility and problem-solving.  
- Reflection fosters self-awareness, helping agents recognize biases and limitations, leading to more informed decision-making.  
- Regular reflection e


## 4. The research node and its fallback

🔮 **Predict first:** with no `TAVILY_API_KEY`, `research_node` falls back to `./corpus`. Each
returned note starts with a **bracketed source tag**. Why does that tag matter downstream — what
does the writer do with it?

In [5]:
hits = _local_corpus_search("Why reflection improves agents")
for h in hits:
    print("-", h[:150], "...")

- [agentic-ai.txt] Common agentic patterns include ReAct (interleaved reasoning and acting), reflection
(the agent critiques and revises its own output) ...
- [reflection.txt] Reflection and Self-Critique in Agents (fictional grounding corpus for Lab 9) ...
- [reflection.txt] Reflection is the pattern where an agent evaluates its own output and revises it
before returning a final answer. A first draft from  ...


## 5. The loop lives on the EDGE, not inside a node

`should_continue` reads State and returns a **label**; the mapping in `add_conditional_edges`
turns that label into the next node. That is the whole reflect/revise cycle.

In [6]:
import inspect
print(inspect.getsource(should_continue))

for revisions, cap in [(0, 2), (1, 2), (2, 2), (3, 2)]:
    label = should_continue({**state, "revisions": revisions, "max_revisions": cap})
    print(f"revisions={revisions} cap={cap} -> {label:<7} -> {'write' if label == 'revise' else 'END'}")

def should_continue(state: EssayState) -> str:
    if state["revisions"] >= state["max_revisions"]:
        return "done"
    return "revise"

revisions=0 cap=2 -> revise  -> write
revisions=1 cap=2 -> revise  -> write
revisions=2 cap=2 -> done    -> END
revisions=3 cap=2 -> done    -> END


## 6. Compile the graph and look at it

In [7]:
graph = build_graph()
g = graph.get_graph()
print("nodes:", [n for n in g.nodes])
print("\nedges:")
for e in g.edges:
    label = f"  [{e.data}]" if getattr(e, "data", None) else ""
    arrow = "..>" if getattr(e, "conditional", False) else "-->"
    print(f"  {e.source:<10} {arrow} {e.target:<10}{label}")
print("\n(The two dotted edges out of `critique` are the conditional pair: revise -> write, done -> END.)")
# `g.draw_ascii()` draws this as a picture, but needs the extra `grandalf` package - not in the
# course lockfile. `g.draw_mermaid()` needs nothing and pastes into any Markdown viewer:
print()
print(g.draw_mermaid())

nodes: ['__start__', 'plan', 'research', 'write', 'critique', '__end__']

edges:
  __start__  --> plan      
  critique   ..> __end__     [done]
  critique   ..> write       [revise]
  plan       --> research  
  research   --> write     
  write      --> critique  

(The two dotted edges out of `critique` are the conditional pair: revise -> write, done -> END.)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	research(research)
	write(write)
	critique(critique)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	critique -. &nbsp;done&nbsp; .-> __end__;
	critique -. &nbsp;revise&nbsp; .-> write;
	plan --> research;
	research --> write;
	write --> critique;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 7. 🔮 Predict, then stream the run

With `max_revisions=2`: how many times does **write** fire, how many times **critique**, and what
finally stops the loop — the cap, or the critic deciding the draft is good enough? Commit to an
answer, then run.

In [8]:
initial: EssayState = {
    "task": "Why reflection improves agents",
    "plan": "", "notes": [], "draft": "", "critique": "",
    "revisions": 0, "max_revisions": 2,
}
config = {"configurable": {"thread_id": "explore-1"}}

for chunk in graph.stream(initial, config):
    for node, update in chunk.items():
        print(f"  -> {node:<9} updated: {list(update)}")

plan: outlining

  -> plan      updated: ['plan']


research: gathering evidence

no TAVILY_API_KEY; using local corpus

  -> research  updated: ['notes']


...

  -> critique  updated: ['critique']


write: drafting (revision 2)

  -> write     updated: ['draft', 'revisions']


critique: reviewing draft

  -> critique  updated: ['critique']


## 8. The checkpointer holds State after every node

State is persisted per `thread_id`, which is what makes a run inspectable (and resumable) rather
than a black box.

In [9]:
final = graph.get_state(config).values
print("revisions:", final["revisions"], "| notes:", len(final["notes"]))
print("\n--- final draft (first 700 chars) ---")
print(final["draft"][:700])

revisions: 2 | notes: 3

--- final draft (first 700 chars) ---
Reflection is a process in which an agent evaluates its own output and revises it before returning a final answer, rather than simply accepting its first response. This approach is part of a broader set of "agentic patterns," which are strategies agents use to improve their performance, including planning (breaking down goals into steps) and multi-agent collaboration (specialized agents working together) [agentic-ai.txt]. Reflection stands out because it directly targets the quality of the agent’s reasoning and output, encouraging the agent to learn from its own mistakes and successes. For example, [reflection.txt] describes how a first draft from a language model is often plausible but shal


## 9. 🛠️ Your turn — let the critic stop the loop early

Right now the loop *only* ends at the cap: a draft the critic is already happy with gets revised
anyway. Make the critic able to end it. Two coordinated edits, exactly as in README step 10:

1. Add to `CRITIQUE_SYS`: *if the draft already meets the rubric, reply with exactly `LGTM` and no
   bullets.*
2. In `should_continue`, after the cap check, return `"done"` when the critique says `LGTM` (or
   contains no bullet lines).

Below is a scratch version you can run **without editing the module** — patch the router, give it
a generous cap, and see whether it stops before the cap. *Reference answer in the README
Teacher's Playbook.*

In [10]:
import essay_writer

essay_writer.CRITIQUE_SYS = essay_writer.CRITIQUE_SYS + (
    " If the draft already meets the rubric, reply with exactly LGTM and no bullets."
)

def should_continue_v2(state: EssayState) -> str:
    if state["revisions"] >= state["max_revisions"]:
        return "done"                                    # cap still wins
    crit = (state.get("critique") or "").strip()
    if crit.upper().startswith("LGTM") or not [l for l in crit.splitlines() if l.strip().startswith(("-", "*"))]:
        return "done"                                    # critic is satisfied
    return "revise"

essay_writer.should_continue = should_continue_v2
graph2 = essay_writer.build_graph()                      # rebuild so the new router is wired in

for chunk in graph2.stream({**initial, "max_revisions": 4},
                           {"configurable": {"thread_id": "explore-lgtm"}}):
    for node, update in chunk.items():
        print(f"  -> {node:<9} updated: {list(update)}")

st = graph2.get_state({"configurable": {"thread_id": "explore-lgtm"}}).values
print("\nstopped after revisions:", st["revisions"], "of cap 4")
print("last critique:", (st["critique"] or "")[:200])

plan: outlining

  -> plan      updated: ['plan']


research: gathering evidence

no TAVILY_API_KEY; using local corpus

  -> research  updated: ['notes']


...

  -> critique  updated: ['critique']


write: drafting (revision 2)

  -> write     updated: ['draft', 'revisions']


critique: reviewing draft

  -> critique  updated: ['critique']

stopped after revisions: 2 of cap 4
last critique: LGTM


## 10. 🗣️ Explain

In two or three sentences: why is the stop decision a **conditional edge** (a router function)
instead of an `if` buried inside `critique_node`? What do you gain by keeping it on the edge —
think about what you can now test, draw, and change without touching a node.